In [1]:
import os
import sys

# Khai báo HADOOP_HOME trỏ tới thư mục chứa bin/winutils.exe
os.environ["HADOOP_HOME"] = "C:\\hadoop"

# Thêm đường dẫn bin vào PATH của hệ thống
os.environ["PATH"] += os.pathsep + "C:\\hadoop\\bin"

print("✅ Đã thiết lập HADOOP_HOME thành công trong môi trường chạy!")

✅ Đã thiết lập HADOOP_HOME thành công trong môi trường chạy!


In [2]:
import os
from pyspark.sql import SparkSession

# Thiết lập đường dẫn Hadoop Home
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["PATH"] += os.pathsep + "C:\\hadoop\\bin"

# Khởi tạo Spark Session cục bộ tối ưu hóa tài nguyên
spark = SparkSession.builder \
    .appName("GDELT-Medallion-EDA") \
    .master("local[*]") \
    .getOrCreate()

# Ẩn bớt các cảnh báo log không cần thiết
spark.sparkContext.setLogLevel("ERROR")

In [3]:
from pyspark.sql.types import StructType, StructField, StringType

gkg_schema = StructType([
    StructField("RecordID", StringType(), True),
    StructField("DateRaw", StringType(), True),
    StructField("SourceCollection", StringType(), True),
    StructField("SourceName", StringType(), True),
    StructField("Url", StringType(), True),
    StructField("Counts", StringType(), True),
    StructField("V2Counts", StringType(), True),
    StructField("Themes", StringType(), True),
    StructField("V2Themes", StringType(), True),
    StructField("Locations", StringType(), True),
    StructField("V2Locations", StringType(), True),
    StructField("Persons", StringType(), True),
    StructField("V2Persons", StringType(), True),
    StructField("Organizations", StringType(), True),
    StructField("V2Organizations", StringType(), True),
    StructField("Tone", StringType(), True),
    StructField("V2EnhancedDates", StringType(), True),
    StructField("V2GCAM", StringType(), True),
    StructField("V2SharingImage", StringType(), True),
    StructField("V2RelatedImages", StringType(), True),
    StructField("V2SocialVideoEmbeds", StringType(), True),
    StructField("V2SocialImageEmbeds", StringType(), True),
    StructField("V2TranslationInfo", StringType(), True),
    StructField("V21ALLNAMES", StringType(), True),
    StructField("V21AMOUNTS", StringType(), True),
    StructField("V21DATES", StringType(), True),
    StructField("V21XMLExtras", StringType(), True)
])

In [4]:
# Thay tên file bằng tên file thực tế trong folder data/ của bạn
file_path = "../data/20260528031500.gkg.csv"

raw_df = spark.read \
    .option("delimiter", "\t") \
    .schema(gkg_schema) \
    .csv(file_path)

# Ghi lưu trữ nguyên bản ra Bronze Lake dạng Parquet
raw_df.write.mode("overwrite").parquet("../data/bronze")
print("✅ Đã ghi dữ liệu nguyên bản vào tầng BRONZE!")

✅ Đã ghi dữ liệu nguyên bản vào tầng BRONZE!


In [5]:
from pyspark.sql import functions as F

# Đọc lại dữ liệu từ tầng Bronze
bronze_df = spark.read.parquet("../data/bronze")

# 1. Trích xuất ToneScore (phần tử đầu tiên trong chuỗi Tone ở cột số 15)
# 2. Định dạng lại kiểu dữ liệu thời gian sang Timestamp chuẩn
# 3. Lọc bỏ các dòng bị NULL ở các cột chính
silver_df = bronze_df.withColumn("ToneScore", F.split(F.col("Tone"), ",")[0].cast("float")) \
                      .withColumn("Date", F.to_timestamp(F.col("DateRaw"), "yyyyMMddHHmmss")) \
                      .filter(F.col("ToneScore").isNotNull() & F.col("Themes").isNotNull()) \
                      .select("RecordID", "Date", "SourceName", "Url", "Themes", "Organizations", "ToneScore", "V2SharingImage")

# Ghi dữ liệu đã làm sạch vào tầng Silver
silver_df.write.mode("overwrite").parquet("../data/silver")
print("✅ Đã xử lý và ghi dữ liệu sạch vào tầng SILVER!")

✅ Đã xử lý và ghi dữ liệu sạch vào tầng SILVER!


In [6]:
# Đọc dữ liệu sạch từ tầng Silver
silver_clean_df = spark.read.parquet("../data/silver")

# 1. Tách chuỗi Themes thành mảng các chủ đề
# 2. explode() mảng chủ đề thành từng dòng độc lập
gold_exploded = silver_clean_df.withColumn("ThemesArray", F.split(F.col("Themes"), ";")) \
                               .withColumn("Theme", F.explode(F.col("ThemesArray"))) \
                               .filter(F.col("Theme") != "")

# 3. Lọc xu hướng kinh tế vĩ mô và tính toán thống kê gom nhóm
gold_econ_trends = gold_exploded.filter(F.col("Theme").startswith("ECON_")) \
                                .groupBy("Theme") \
                                .agg(
                                    F.count("RecordID").alias("ArticleCount"),
                                    F.round(F.avg("ToneScore"), 2).alias("AvgTone")
                                 ) \
                                .filter(F.col("ArticleCount") >= 5) \
                                .orderBy(F.desc("ArticleCount"))

# Ghi dữ liệu curated vào tầng Gold
gold_econ_trends.write.mode("overwrite").parquet("../data/gold")
print("✅ Đã phân tích xu hướng và ghi vào tầng GOLD!")

# Chiêm ngưỡng bảng kết quả đẹp mắt trên Jupyter
gold_econ_trends.limit(10).toPandas()

✅ Đã phân tích xu hướng và ghi vào tầng GOLD!


,Theme,ArticleCount,AvgTone
0,ECON_TAXATION,39,-2.31
1,ECON_WORLDCURRENCIES,32,-1.23
2,ECON_STOCKMARKET,29,-1.07
3,ECON_HOUSING_PRICES,23,-1.35
4,ECON_OILPRICE,21,-1.63
5,ECON_INFLATION,19,-2.02
6,ECON_BITCOIN,17,-2.37
7,ECON_WORLDCURRENCIES_DOLLARS,16,-2.13
8,ECON_ENTREPRENEURSHIP,15,-1.54
9,ECON_BUDGET_DEFICIT,14,-2.30
